In [16]:

import os
import json
import time
import re
from tavily import TavilyClient

client = TavilyClient("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")

SYSTEM_INSTRUCTION = """
Determine the correct information for this music event
from current web evidence. Some links to the event are provided, but you may also search the web for additional sources.

Return the official event title, date, venue, city,
description, and all confirmed performing artists, including support artists, find as many of these as possible as they may not all be listed on every event page.

For each artist:
- Find their official YouTube Music artist page.
- Search music.youtube.com for the artist's exact name.
- Prefer the official artist page or Topic channel.
- Return the full artist page URL, not just an ID.
- Accept music.youtube.com/@... or music.youtube.com/channel/... URLs.
- Do not return links to songs, videos, albums,
  playlists, or search result pages.
- If you cannot find a decent match for the artist's official YouTube Music page,
  return an empty string.


Use YYYY-MM-DD for dates.
Include one entry per confirmed performing artist.
Do not invent missing information.
"""


schema = {
    "properties": {
        "event_title": {
            "type": "string",
            "description": "Official event title"
        },
        "event_date": {
            "type": "string",
            "description": "Event date in YYYY-MM-DD format"
        },
        "venue": {
            "type": "string",
            "description": "Official venue name"
        },
        "city": {
            "type": "string",
            "description": "City where the event takes place"
        },
        "description": {
            "type": "string",
            "description": "Factual event description"
        },
        "artists": {
            "type": "array",
            "description": "All confirmed performing artists",
            "items": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "Official artist name"
                    },
                    "youtube_music_url": {
                        "type": "string",
                        "description": (
                             "Verified direct URL to the artist's "
                            "YouTube Music artist page. "
                            "Use an empty string if unavailable."
                        )
                    }
                },
                "required": ["name", "youtube_music_url"]
            }
        }
    },
    "required": [
        "event_title", "event_date", "venue",
        "city", "description", "artists"
    ]
}

event = "SHAKE SOME ACTON HALLOWEEN TRASH ROCK N ROLL PARTY! Signature Brew Haggerston london 2026-10-30 https://dice.fm/event/bb9vvo-shake-some-acton-halloween-trash-rock-n-roll-party-30th-oct-signature-brew-haggerston-london-tickets https://www.songkick.com/concerts/43311491-darts-us-at-signature-brew-haggerston https://www.bandsintown.com/e/1039696438-mystery-action-at-signature-brew-haggerston"

task = client.research(
    input=f"{SYSTEM_INSTRUCTION}\n\nEvent: {event}",
    model="mini",
    output_schema=schema
)

request_id = task["request_id"]

# Research runs asynchronously
for _ in range(120):
    result = client.get_research(
        request_id,
        include_usage=True
    )

    if result["status"] == "completed":
        break

    if result["status"] == "failed":
        raise RuntimeError(result.get("error", "Research failed"))

    time.sleep(5)
else:
    raise TimeoutError(f"Research still running: {request_id}")

content = result["content"]
data = json.loads(content) if isinstance(content, str) else content

# Convert unknown strings to JSON null
def nullable(value):
    return value.strip() if isinstance(value, str) and value.strip() else None

for field in [
    "event_title", "event_date", "venue", "city", "description"
]:
    data[field] = nullable(data.get(field))

for artist in data.get("artists", []):
    artist["name"] = nullable(artist.get("name"))

    youtube_music_url = nullable(artist.get("youtube_music_url"))

    # Structural check only, not proof of artist identity
    if not youtube_music_url: 
        youtube_music_url = None
    
        

    artist["youtube_music_url"] = youtube_music_url

print(json.dumps(data, indent=2, ensure_ascii=False))


{
  "event_title": "SHAKE SOME ACTON HALLOWEEN TRASH ROCK N ROLL PARTY!",
  "event_date": "2026-10-30",
  "venue": "Signature Brew Haggerston",
  "city": "London",
  "description": "A Halloween‑themed garage rock and punk party at Signature Brew Haggerston in London on 30 October 2026. The event, titled SHAKE SOME ACTON HALLOWEEN TRASH ROCK N ROLL PARTY!, features US garage rockers The Darts, primal garage‑punk outfit WitchDoktors, and Midlands rockers Mystery Action. Attendees are encouraged to wear spooky costumes for the Samhain celebration.",
  "artists": [
    {
      "name": "The Darts",
      "youtube_music_url": "https://music.youtube.com/channel/UCrxMOsGxG2wuiTm0ted8cmw"
    },
    {
      "name": "WitchDoktors",
      "youtube_music_url": null
    },
    {
      "name": "Mystery Action",
      "youtube_music_url": null
    }
  ]
}
